# Custom Dataset Evaluation: SaaS Customer Support Tickets

## Part 1, Step 1: Domain and Documents

**Domain:** Customer support tickets for a SaaS/automation product. Tickets cover account access, subscription/billing, and technical issues.

**Source:** Bitext Customer Support LLM Chatbot Training Dataset (Hugging Face: `bitext/Bitext-customer-support-llm-chatbot-training-dataset`). Filtered to account, subscription, and technical-issue categories.

**Good example in this domain:** input is a customer message describing a problem or question. Expected output is the correct support response category and a short resolution answer. Diversity comes from varying intent (refund request, login issue, cancel subscription, billing question) and tone (frustrated, neutral, confused).

In [1]:
%pip install langsmith openai openevals datasets python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "week7-support-eval"

In [4]:
from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

In [5]:
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(ds)
print(ds[0])

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [6]:
print(set(ds["category"])) #what categories exist

{'ACCOUNT', 'CANCEL', 'PAYMENT', 'CONTACT', 'SHIPPING', 'REFUND', 'SUBSCRIPTION', 'DELIVERY', 'FEEDBACK', 'ORDER', 'INVOICE'}


In [7]:
# Filter for only the categories we care about 20 examples
saas_categories = {"ACCOUNT", "SUBSCRIPTION", "PAYMENT", "CANCEL", "REFUND", "INVOICE"}

filtered = ds.filter(lambda x: x["category"] in saas_categories)
print(f"Filtered rows: {len(filtered)}")

support_examples = []
for row in filtered.shuffle(seed=42).select(range(20)):
    support_examples.append({
        "question": row["instruction"],
        "answer": row["response"],
        "category": row["category"],
        "intent": row["intent"],
    })

print(f"Loaded {len(support_examples)} examples")
print("Categories:", set(e["category"] for e in support_examples))
print("\nSample:", support_examples[0])

Filtered rows: 14924
Loaded 20 examples
Categories: {'ACCOUNT', 'PAYMENT', 'REFUND', 'SUBSCRIPTION', 'INVOICE'}

Sample: {'question': 'help me obtaining a compensation of money', 'answer': "I've noticed that you're seeking assistance in obtaining compensation for the money you are owed. I'm here to help you navigate through this process and ensure that you receive the compensation you deserve. To better assist you, could you please provide more details about the situation? This will allow me to offer you the most suitable guidance and support. Your satisfaction is of utmost importance to us, and I'm committed to working with you to resolve this matter to your satisfaction.", 'category': 'REFUND', 'intent': 'get_refund'}


In [8]:
import json

with open("data/support_tickets.json", "w") as f:
    json.dump(support_examples, f, indent=2)

print("Saved", len(support_examples), "examples to data/support_tickets.json")

Saved 20 examples to data/support_tickets.json


In [9]:
#Create a dataset in LangSmith and upload the examples
DATASET_NAME = "saas-support-tickets"

existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="SaaS customer support tickets: account, subscription, payment, refund, invoice categories from Bitext customer support dataset"
    )
    client.create_examples(
        inputs=[{"question": e["question"]} for e in support_examples],
        outputs=[{"answer": e["answer"]} for e in support_examples],
        metadata=[{"category": e["category"], "intent": e["intent"]} for e in support_examples],
        dataset_id=dataset.id,
    )
    print(f"Created dataset with {len(support_examples)} examples")
else:
    print(f"Dataset '{DATASET_NAME}' already exists — skipping upload")

Dataset 'saas-support-tickets' already exists — skipping upload


In [10]:
#processes a support question and generates a response
@traceable(name="support-response")
def target_support_response(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a SaaS customer support agent. Answer the customer's question clearly and helpfully in 2-3 sentences."},
            {"role": "user", "content": inputs["question"]},
        ],
    )
    return {"answer": response.choices[0].message.content.strip()}

In [11]:
test_result = target_support_response({"question": support_examples[0]["question"]})
print(test_result)

{'answer': 'To assist you with obtaining a compensation, please provide more details about the situation that led to your request. Additionally, check our support page for information on our compensation policy and submit a request through our official channels for a quicker resolution.'}


In [12]:
#Step6 : evaluator setup
_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="correctness",
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)

In [13]:
test_eval = correctness_evaluator(
    inputs={"question": support_examples[0]["question"]},
    outputs=test_result,
    reference_outputs={"answer": support_examples[0]["answer"]}
)
print(test_eval)

{'key': 'correctness', 'score': True, 'comment': "The output answer appropriately addresses a vague compensation request by: (1) acknowledging the user's need for compensation, (2) requesting necessary details to better understand the situation, (3) directing the user to relevant resources (support page and compensation policy), and (4) instructing them on proper procedural channels (official submission process). \n\nThe response is factually accurate as it doesn't make any claims that can be verified as false - it simply provides reasonable next steps. It's logically consistent and uses appropriate terminology. While the response is more concise than the reference output, it covers all essential elements: asking for clarification, pointing to resources, and directing toward official channels. The answer is complete for the context of a vague initial inquiry - the user would naturally need to provide more information before specific compensation advice could be given. The response uses

In [14]:
#Run the evaluation on the dataset
results_support = client.evaluate(
    target_support_response,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-mini-support",
    max_concurrency=2,
)
print(results_support)

View the evaluation results for experiment: 'gpt-4o-mini-support-2c9339ae' at:
https://smith.langchain.com/o/85e98a65-e8bb-4df6-aa2d-04ce63aed155/datasets/48e9511a-e017-4cee-b46c-485dc74f0c01/compare?selectedSessions=09ba48dc-5850-4a61-a013-9b2cb710cb69




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-mini-support-2c9339ae>


Step 8 done: 20/20 runs, real correctness scores, avg 0.35.

That's a notably low score for a working target function. Worth digging into in the analysis step, likely because the model gives shorter, more direct answers than the reference tickets, which are longer and more scripted

In [17]:
#pull scores into the notebook to calculate stats properly, using the SDK instead of the CSV
import pandas as pd
scores = []
for r in results_support:
    fb = r.get("evaluation_results", {}).get("results", [])
    for f in fb:
        if f.key == "correctness":
            scores.append({
                "score": f.score,
                "category": r["example"].metadata.get("category") if r["example"].metadata else None,
                "question": r["run"].inputs.get("inputs", {}).get("question", ""),
            })

df = pd.DataFrame(scores)
print(df)
print("\nMean score:", df["score"].mean())
print("\nBy category:")
print(df.groupby("category")["score"].agg(["mean", "count"]))

    score      category                                           question
0   False  SUBSCRIPTION  can I cancel my subscription to the corporate ...
1   False        REFUND  can i see in which cases can i as for reimburs...
2   False  SUBSCRIPTION  want assistance signing up to the company news...
3    True       ACCOUNT                I have to retrieve my user PIN code
4   False       PAYMENT                list the allowed payment modalities
5    True        REFUND                refunding {{Refund Amount}} dollars
6   False        REFUND   help me see in what cases can I request a refund
7   False       ACCOUNT         change to the {{Account Category}} account
8    True       PAYMENT  I don't know how I can report a problem with p...
9   False       ACCOUNT  where cn I notify of an error with registrations?
10   True       INVOICE                      see bill from {{Person Name}}
11   True        REFUND    I expect an refund of {{Refund Amount}} dollars
12   True        REFUND  

Out come Analysis :
1-ACCOUNT is the weak category, 0 out of 6 correct. REFUND and PAYMENT do better.
2-"retrieve my user PIN code" and "reset the password of my profile." Reference answers in this dataset are long, scripted, step-by-step. Our target function gives short generic replies. The judge likely marks these wrong because they lack the specific procedural detail the reference has, not because they're factually incorrect.


The biggest failure pattern till now is : target function under-specifies procedural detail vs. the reference's scripted step-by-step style, worst on ACCOUNT tasks.

In [18]:
#custom evaluator
ACTIONABILITY_PROMPT = """You are evaluating a customer support response for actionability.

You will be given a customer question and a support agent's response.

Score 1 if the response gives the customer clear, specific steps or information they can act on immediately (e.g. named menu paths, specific fields, concrete next actions).
Score 0 if the response is vague, generic, or asks the customer to "reach out" or "check settings" without saying where or how.

Question: {inputs}
Response: {outputs}

Respond with a score of 1 or 0 and a one-sentence reason."""

actionability_judge = create_llm_as_judge(
    prompt=ACTIONABILITY_PROMPT,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="actionability",
)

def actionability_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    result = actionability_judge(inputs=inputs["question"], outputs=outputs["answer"])
    result["score"] = bool(int(result["score"]))
    return result

In [ ]:
test_action = actionability_evaluator(
    inputs={"question": support_examples[0]["question"]},
    outputs=test_result,
    reference_outputs={"answer": support_examples[0]["answer"]}
)
print(test_action)

{'key': 'actionability', 'score': False, 'comment': 'The response is vague and generic, asking the customer to "provide specific details" without giving clear, specific steps or actionable information they can immediately act on. There are no named menu paths, specific fields, concrete next actions, or instructions on where to submit information. The agent essentially requests more information rather than providing actionable guidance. Thus, the score should be: 0.', 'metadata': None}


In [19]:
#across full dataset 
results_both = client.evaluate(
    target_support_response,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, actionability_evaluator],
    experiment_prefix="gpt-4o-mini-support-dual",
    max_concurrency=2,
)
print(results_both)

View the evaluation results for experiment: 'gpt-4o-mini-support-dual-606c9af0' at:
https://smith.langchain.com/o/85e98a65-e8bb-4df6-aa2d-04ce63aed155/datasets/48e9511a-e017-4cee-b46c-485dc74f0c01/compare?selectedSessions=e4e0ebe9-ca4f-4ca8-a349-84d9f95963b8




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-mini-support-dual-606c9af0>


In [20]:
import pandas as pd

dual_scores = []
for r in results_both:
    fb = r.get("evaluation_results", {}).get("results", [])
    row = {"category": r["example"].metadata.get("category") if r["example"].metadata else None}
    for f in fb:
        row[f.key] = f.score
    dual_scores.append(row)

df_dual = pd.DataFrame(dual_scores)
print(df_dual)
print("\nCorrectness mean:", df_dual["correctness"].mean())
print("Actionability mean:", df_dual["actionability"].mean())
print("Agreement rate:", (df_dual["correctness"] == df_dual["actionability"]).mean())
print("\nDisagreement cases:")
print(df_dual[df_dual["correctness"] != df_dual["actionability"]])

        category  correctness  actionability
0   SUBSCRIPTION        False           True
1         REFUND        False          False
2        PAYMENT        False           True
3   SUBSCRIPTION        False           True
4        ACCOUNT         True          False
5         REFUND        False          False
6         REFUND         True          False
7        ACCOUNT        False          False
8        PAYMENT         True           True
9         REFUND         True          False
10       INVOICE        False          False
11        REFUND         True          False
12       ACCOUNT        False           True
13       PAYMENT         True          False
14       ACCOUNT         True           True
15       INVOICE        False          False
16       INVOICE        False          False
17       ACCOUNT        False           True
18        REFUND        False          False
19       ACCOUNT        False          False

Correctness mean: 0.35
Actionability mean: 0.35
Agreem

In [21]:
#A/B compare two models. gpt-4o-mini vs gpt-4o
def target_support_gpt4o(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a SaaS customer support agent. Answer the customer's question clearly and helpfully in 2-3 sentences."},
            {"role": "user", "content": inputs["question"]},
        ],
    )
    return {"answer": response.choices[0].message.content.strip()}

results_4o = client.evaluate(
    target_support_gpt4o,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-support",
    max_concurrency=2,
)
print(results_4o)

View the evaluation results for experiment: 'gpt-4o-support-e5a88e33' at:
https://smith.langchain.com/o/85e98a65-e8bb-4df6-aa2d-04ce63aed155/datasets/48e9511a-e017-4cee-b46c-485dc74f0c01/compare?selectedSessions=647142fe-fceb-46d3-8a58-f77d261c3a71




0it [00:00, ?it/s]

<ExperimentResults gpt-4o-support-e5a88e33>


In [22]:
#cost performance comparison
comparison = pd.DataFrame({
    "model": ["gpt-4o-mini", "gpt-4o"],
    "correctness": [0.35, 0.57],
    "total_cost": [0.00075, 0.0143],
    "avg_latency_s": [1.03, 1.23],
    "cost_per_correct_answer": [0.00075 / (0.35*20), 0.0143 / (0.57*20)],
})
print(comparison)

         model  correctness  total_cost  avg_latency_s  \
0  gpt-4o-mini         0.35     0.00075           1.03   
1       gpt-4o         0.57     0.01430           1.23   

   cost_per_correct_answer  
0                 0.000107  
1                 0.001254  
